# MARL Coordination Demo — PPO vs MAPPO vs MADDPG

**Interactive UI for simple_spread_v3**

- **Cell 1** — Install dependencies
- **Cell 2** — Imports
- **Cell 3** — Model architectures (must match training)
- **Cell 4** — Simulation policies (no .pth needed)
- **Cell 5** — Real model loader (swap in your .pth files)
- **Cell 6** — Environment runner
- **Cell 7** — LAUNCH UI ← run this to open the demo

> **Quick start:** Run all cells top to bottom, then interact with Cell 7.

In [6]:
# ============================================================
# CELL 1 — Install
# ============================================================
!pip install -q pettingzoo==1.24.3 gymnasium==0.29.1 supersuit==3.9.3 torch numpy matplotlib ipywidgets
from google.colab import output
output.enable_custom_widget_manager()
print('✅ Installation complete')

✅ Installation complete


In [7]:
# ============================================================
# CELL 2 — Imports
# ============================================================
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation
import ipywidgets as widgets
from IPython.display import display, clear_output
import time, os
from pettingzoo.mpe import simple_spread_v3

print('✅ Imports complete')

✅ Imports complete


In [8]:
# ============================================================
# CELL 3 — Model Architectures (must match training exactly)
# ============================================================

class PPOMAPPOActor(nn.Module):
    """Shared actor for PPO and MAPPO — discrete actions, Softmax"""
    def __init__(self, obs_dim=18, act_dim=5, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),  nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, act_dim)
        )
    def forward(self, x):
        return torch.softmax(self.net(x), dim=-1)
    def act(self, obs_np):
        with torch.no_grad():
            x = torch.FloatTensor(obs_np).unsqueeze(0)
            return self.forward(x).argmax(dim=-1).item()

class MADDPGActor(nn.Module):
    """Per-agent actor for MADDPG — continuous actions, Sigmoid"""
    def __init__(self, obs_dim=18, act_dim=5, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),  nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, act_dim), nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)
    def act(self, obs_np):
        with torch.no_grad():
            x = torch.FloatTensor(obs_np).unsqueeze(0)
            return self.forward(x).squeeze(0).numpy()

class RunningNorm:
    def __init__(self, shape, clip=5.0, eps=1e-8):
        self.mean  = np.zeros(shape, np.float64)
        self.var   = np.ones(shape,  np.float64)
        self.count = eps
        self.clip  = clip
        self.eps   = eps
    def update(self, x):
        x = np.asarray(x, np.float64)
        self.count += 1
        d = x - self.mean
        self.mean += d / self.count
        self.var  += d * (x - self.mean)
    @property
    def std(self):
        return np.sqrt(self.var / self.count + self.eps)
    def normalize(self, x):
        return np.clip(
            (np.asarray(x, np.float64) - self.mean) / self.std,
            -self.clip, self.clip
        ).astype(np.float32)

print('✅ Model architectures defined')

✅ Model architectures defined


In [9]:
# ============================================================
# CELL 4 — Simulation Policies (no .pth files needed)
# These simulate the LEARNED behaviour of each algorithm
# ============================================================

AGENTS = ['agent_0', 'agent_1', 'agent_2']

def sim_ppo_actions(obs_dict):
    """PPO simulation: each agent greedily targets nearest landmark → crowding"""
    # Discrete action map: 0=stay, 1=left, 2=right, 3=down, 4=up
    actions = {}
    for ag in AGENTS:
        obs = obs_dict[ag]
        # obs[2:4] = self pos, obs[4:10] = landmark rel pos
        lm_rel = [(obs[4+l*2], obs[5+l*2]) for l in range(3)]
        # Greedy: go to nearest landmark
        dists  = [np.sqrt(r[0]**2+r[1]**2) for r in lm_rel]
        target = np.argmin(dists)
        dx, dy = lm_rel[target]
        if   abs(dx) > abs(dy): actions[ag] = 2 if dx > 0 else 1
        else:                   actions[ag] = 4 if dy > 0 else 3
    return actions

def sim_mappo_actions(obs_dict):
    """MAPPO simulation: greedy assignment — each agent takes unique landmark"""
    actions = {}
    taken   = set()
    # Priority by agent index — mimics centralised critic assignment
    for i, ag in enumerate(AGENTS):
        obs    = obs_dict[ag]
        lm_rel = [(obs[4+l*2], obs[5+l*2]) for l in range(3)]
        dists  = [np.sqrt(r[0]**2+r[1]**2) if l not in taken else 999
                  for l, r in enumerate(lm_rel)]
        target = np.argmin(dists)
        taken.add(target)
        dx, dy = lm_rel[target]
        if   abs(dx) > abs(dy): actions[ag] = 2 if dx > 0 else 1
        else:                   actions[ag] = 4 if dy > 0 else 3
    return actions

def sim_maddpg_actions(obs_dict, agents_last_obs=None):
    """MADDPG simulation: competition-aware assignment, continuous actions [0,1]^5"""
    # MADDPG uses Box(0.0, 1.0, (5,)) — must return float32 array, NOT int
    # Action format: [no_action, move_left, move_right, move_down, move_up]
    actions = {}
    for i, ag in enumerate(AGENTS):
        obs    = obs_dict[ag]
        lm_rel = [(obs[4+l*2], obs[5+l*2]) for l in range(3)]
        # Score each landmark: distance + competitor penalty
        scores = []
        for l, (rx, ry) in enumerate(lm_rel):
            self_d = np.sqrt(rx**2+ry**2)
            comp = 0
            for j, other in enumerate(AGENTS):
                if j == i: continue
                other_obs = obs_dict[other]
                other_lm  = (other_obs[4+l*2], other_obs[5+l*2])
                other_d   = np.sqrt(other_lm[0]**2+other_lm[1]**2)
                if other_d < self_d: comp += (self_d - other_d) * 0.85
            scores.append(self_d + comp)
        target = np.argmin(scores)
        dx, dy = lm_rel[target]
        # Build continuous action vector — one-hot style with magnitude
        act = np.zeros(5, dtype=np.float32)
        act[0] = 0.0  # no-op
        dist = np.sqrt(dx**2 + dy**2) + 1e-6
        # Scale thrust by distance (closer = gentler)
        thrust = float(np.clip(dist * 0.8, 0.3, 1.0))
        if abs(dx) > abs(dy):
            act[2 if dx > 0 else 1] = thrust  # right or left
        else:
            act[4 if dy > 0 else 3] = thrust  # up or down
        actions[ag] = act
    return actions

# Map algo name to simulation function
SIM_POLICIES = {
    'PPO':    sim_ppo_actions,
    'MAPPO':  sim_mappo_actions,
    'MADDPG': sim_maddpg_actions,
}

print('✅ Simulation policies ready')

✅ Simulation policies ready


In [10]:
# ============================================================
# CELL 5 — Real Model Loader
# Set USE_REAL_MODELS = True and update MODEL_DIRS to use
# your actual trained .pth files from Google Drive
# ============================================================

USE_REAL_MODELS = False # for PPO and MAPPO
# USE_REAL_MODELS = True   # for MADDPG ← Change to True when .pth files are ready

# Update these paths to your Google Drive model locations
MODEL_DIRS = {
    'PPO':    '/content/drive/MyDrive/PES Individual projects/PPO/PPO_1M',
    'MAPPO':  '/content/drive/MyDrive/PES Individual projects/MAPPO/MAPPO_1p5M',
    'MADDPG': '/content/drive/MyDrive/PES Individual projects/MADDPG/MADDPG_1.5_results/MADDPG_1p5M',
}

real_actors = {}   # populated below if USE_REAL_MODELS = True

if USE_REAL_MODELS:
    from google.colab import drive
    drive.mount('/content/drive')

    for algo, model_dir in MODEL_DIRS.items():
        actors = []
        continuous = (algo == 'MADDPG')
        ActorClass = MADDPGActor if continuous else PPOMAPPOActor
        ok = True
        for i in range(3):
            path = os.path.join(model_dir, f'best_actor_{i}.pth')
            if not os.path.exists(path):
                print(f'  ⚠ {algo} actor {i} not found at {path}')
                ok = False
                break
            a = ActorClass()
            a.load_state_dict(torch.load(path, map_location='cpu'))
            a.eval()
            actors.append(a)
        if ok:
            real_actors[algo] = actors
            print(f'  ✅ {algo} models loaded from {model_dir}')

    def make_real_policy(algo):
        actors    = real_actors[algo]
        norms     = [RunningNorm(18) for _ in range(3)]
        continuous = (algo == 'MADDPG')
        env_disc  = simple_spread_v3.parallel_env(
            N=3, local_ratio=0.5, max_cycles=50,
            continuous_actions=continuous
        )
        def policy(obs_dict):
            actions = {}
            for i, ag in enumerate(AGENTS):
                norms[i].update(obs_dict[ag])
                norm_obs = norms[i].normalize(obs_dict[ag])
                actions[ag] = actors[i].act(norm_obs)
            return actions
        return policy

    REAL_POLICIES = {algo: make_real_policy(algo)
                     for algo in real_actors}
    print('\n✅ Real model policies ready:', list(REAL_POLICIES.keys()))
else:
    REAL_POLICIES = {}
    print('ℹ  Simulation mode — set USE_REAL_MODELS=True to load .pth files')

ℹ  Simulation mode — set USE_REAL_MODELS=True to load .pth files


In [11]:
# ============================================================
# CELL 6 — Environment Runner
# ============================================================

ALGO_COLORS = {
    'PPO':    ['#378ADD', '#1D9E75', '#D85A30'],
    'MAPPO':  ['#378ADD', '#1D9E75', '#D85A30'],
    'MADDPG': ['#378ADD', '#1D9E75', '#D85A30'],
}
ALGO_ACCENT = {'PPO':'#185FA5','MAPPO':'#028090','MADDPG':'#534AB7'}
ZONE_COLOR  = '#BA7517'

ALGO_INFO = {
    'PPO':   {
        'badge': 'Independent · local critic 18-dim',
        'finding': 'Credit assignment problem — local critic cannot see teammates → crowding → worst episode −303.95',
        'result': 'Eval mean: −47.23  |  Std: 29.31  |  Best avg100: −42.72'
    },
    'MAPPO': {
        'badge': 'CTDE · centralised critic 54-dim',
        'finding': 'Centralised critic resolves credit assignment → 7× lower variance (std ~4.2 at 1M) → best eval mean',
        'result': 'Eval mean: −40.60  |  Std: ~4.2  |  Best single episode: −9.85'
    },
    'MADDPG': {
        'badge': 'CTDE · off-policy · critic 69-dim state+actions',
        'finding': 'Off-policy replay reuses 1M transitions per update → best training performance across all 9 runs',
        'result': 'Best avg100: −31.69 (overall best)  |  MADDPG 1.5M eval mean: −43.55'
    }
}

class EpisodeRunner:
    def __init__(self, algo, use_real=False):
        self.algo       = algo
        self.continuous = (algo == 'MADDPG')
        self.env = simple_spread_v3.parallel_env(
            N=3, local_ratio=0.5, max_cycles=50,
            continuous_actions=self.continuous
        )
        if use_real and algo in REAL_POLICIES:
            self.policy    = REAL_POLICIES[algo]
            self.mode_label= 'Real model'
        else:
            self.policy    = SIM_POLICIES[algo]
            self.mode_label= 'Simulation'
        self.reset()

    def reset(self):
        self.obs, _      = self.env.reset()
        self.step_n      = 0
        self.total_reward= 0.0
        self.done        = False
        self.history     = []   # list of (agent_positions, landmark_positions)
        self.redundancies= 0
        self._record()

    def _record(self):
        pos = {}
        for ag in AGENTS:
            o = self.obs[ag]
            pos[ag] = (float(o[2]), float(o[3]))
        # Extract landmark positions from agent_0 obs
        o0  = self.obs[AGENTS[0]]
        ax, ay = float(o0[2]), float(o0[3])
        lms = [(ax+float(o0[4+l*2]), ay+float(o0[5+l*2])) for l in range(3)]
        self.history.append({'pos': pos, 'lms': lms,
                              'reward': self.total_reward,
                              'step': self.step_n})

    def step(self):
        if self.done: return
        actions = self.policy(self.obs)
        self.obs, rewards, terms, truncs, _ = self.env.step(actions)
        self.total_reward += float(np.mean(list(rewards.values())))
        self.step_n += 1
        # Count redundancies
        o0   = self.obs[AGENTS[0]]
        ax0, ay0 = float(o0[2]), float(o0[3])
        lms  = [(ax0+float(o0[4+l*2]), ay0+float(o0[5+l*2])) for l in range(3)]
        apos = {}
        for ag in AGENTS:
            oa = self.obs[ag]
            apos[ag] = (float(oa[2]), float(oa[3]))
        for lx, ly in lms:
            cnt = sum(1 for ag in AGENTS
                      if np.sqrt((apos[ag][0]-lx)**2+(apos[ag][1]-ly)**2)<0.2)
            if cnt > 1: self.redundancies += cnt-1
        self.done = all(terms.values()) or all(truncs.values())
        self._record()

    def run_full(self):
        while not self.done:
            self.step()

    def coverage(self):
        if not self.history: return 0
        snap = self.history[-1]
        covered = 0
        for lx,ly in snap['lms']:
            for ag in AGENTS:
                ax,ay = snap['pos'][ag]
                if np.sqrt((ax-lx)**2+(ay-ly)**2)<0.2:
                    covered+=1; break
        return covered

print('✅ Environment runner ready')

✅ Environment runner ready


In [12]:
# ============================================================
# CELL 7 — LAUNCH UI
# ============================================================

# ── State ────────────────────────────────────────────────────
state = {
    'algo':   'PPO',
    'runner': None,
    'use_real': False,
    'running': False,
    'episode_rewards': [],
}

# ── Widgets ───────────────────────────────────────────────────
algo_toggle = widgets.ToggleButtons(
    options=['PPO', 'MAPPO', 'MADDPG'],
    value='PPO',
    style={'button_width': '150px'},
    layout=widgets.Layout(margin='0 0 8px 0')
)
model_toggle = widgets.ToggleButtons(
    options=['Simulation', 'Real model'],
    value='Simulation',
    style={'button_width': '120px'},
    layout=widgets.Layout(margin='0 0 8px 0')
)
btn_new    = widgets.Button(description='▶ New episode',
    button_style='primary', layout=widgets.Layout(width='140px'))
btn_step   = widgets.Button(description='→ Step',
    layout=widgets.Layout(width='90px'))
btn_run    = widgets.Button(description='⏩ Run full',
    layout=widgets.Layout(width='110px'))
btn_eval   = widgets.Button(description='📊 Eval 50 eps',
    button_style='info', layout=widgets.Layout(width='130px'))
step_slider= widgets.IntSlider(
    min=0, max=50, value=0, step=1,
    description='Step:',
    style={'description_width':'40px'},
    layout=widgets.Layout(width='300px')
)
speed_slider = widgets.FloatSlider(
    min=0.05, max=1.0, value=0.3, step=0.05,
    description='Delay:',
    style={'description_width':'45px'},
    layout=widgets.Layout(width='250px'),
    readout_format='.2f'
)

# Output areas
out_canvas  = widgets.Output()
out_metrics = widgets.Output()
out_info    = widgets.Output()
out_eval    = widgets.Output()

# ── Drawing ───────────────────────────────────────────────────
def draw_frame(snap, algo, step_n, total_r, redundancies):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    fig.patch.set_facecolor('#F8F9FC')

    colors = ALGO_COLORS[algo]
    accent = ALGO_ACCENT[algo]

    # ── Left: environment canvas ──────────────────────────────
    ax = axes[0]
    ax.set_facecolor('#FFFFFF')
    ax.set_xlim(-1.8, 1.8)
    ax.set_ylim(-1.8, 1.8)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.08, color='gray', lw=0.4)
    ax.set_title(f'{algo} — Step {step_n}/50',
                 color=accent, fontsize=13, fontweight='bold', pad=8)

    # Landmarks
    for l, (lx, ly) in enumerate(snap['lms']):
        # Coverage circle
        covered = any(
            np.sqrt((snap['pos'][ag][0]-lx)**2+(snap['pos'][ag][1]-ly)**2)<0.2
            for ag in AGENTS
        )
        circle = plt.Circle((lx,ly), 0.2,
            color=colors[l] if covered else ZONE_COLOR,
            alpha=0.15, zorder=1)
        ax.add_patch(circle)
        ax.add_patch(mpatches.FancyBboxPatch(
            (lx-0.12, ly-0.12), 0.24, 0.24,
            boxstyle='round,pad=0.03',
            facecolor=colors[l] if covered else ZONE_COLOR,
            edgecolor='white', linewidth=1.5, zorder=3
        ))
        ax.text(lx, ly, f'Z{l+1}', ha='center', va='center',
                fontsize=10, fontweight='bold', color='white', zorder=4)
        if covered:
            ax.text(lx, ly+0.22, '✓', ha='center', va='bottom',
                    fontsize=10, color=colors[l], zorder=4)
        # Overlap warning for PPO
        if algo == 'PPO':
            cnt = sum(1 for ag in AGENTS
                      if np.sqrt((snap['pos'][ag][0]-lx)**2+(snap['pos'][ag][1]-ly)**2)<0.2)
            if cnt > 1:
                warn = plt.Circle((lx,ly), 0.28,
                    fill=False, edgecolor='#E24B4A',
                    linewidth=2, linestyle='--', zorder=5)
                ax.add_patch(warn)
                ax.text(lx, ly+0.32, 'OVERLAP!', ha='center',
                        fontsize=9, color='#E24B4A', fontweight='bold', zorder=6)

    # Agents
    for i, ag in enumerate(AGENTS):
        ax_, ay_ = snap['pos'][ag]
        # Target line to nearest landmark
        nl, nd = -1, 999
        for l, (lx,ly) in enumerate(snap['lms']):
            d = np.sqrt((ax_-lx)**2+(ay_-ly)**2)
            if d < nd: nd=d; nl=l
        if nl>=0 and nd>0.2:
            lx, ly = snap['lms'][nl]
            ax.plot([ax_,lx],[ay_,ly],
                    color=colors[i], alpha=0.3, lw=1,
                    linestyle='--', zorder=2)
        # Robot circle
        robot = plt.Circle((ax_,ay_), 0.13,
            facecolor=colors[i], edgecolor='white',
            linewidth=2, zorder=6)
        ax.add_patch(robot)
        ax.text(ax_, ay_, f'R{i+1}', ha='center', va='center',
                fontsize=9, fontweight='bold', color='white', zorder=7)

    # Legend
    legend_elems = [
        mpatches.Patch(color=colors[i], label=f'Robot {i+1}') for i in range(3)
    ] + [mpatches.Patch(color=ZONE_COLOR, label='Zone (unserved)')]
    ax.legend(handles=legend_elems, loc='lower right',
              fontsize=8, framealpha=0.8)

    # Progress bar
    ax.add_patch(mpatches.FancyBboxPatch(
        (-1.75, -1.75), 3.5*(step_n/50), 0.08,
        boxstyle='round,pad=0.01',
        facecolor=accent, alpha=0.7, zorder=8
    ))

    # ── Right: metrics panel ──────────────────────────────────
    ax2 = axes[1]
    ax2.set_facecolor('#F8F9FC')
    ax2.axis('off')

    info = ALGO_INFO[algo]
    cov  = state['runner'].coverage() if state['runner'] else 0

    # Title
    ax2.text(0.05, 0.97, algo, fontsize=18, fontweight='bold',
             color=accent, transform=ax2.transAxes, va='top')
    ax2.text(0.05, 0.91, info['badge'], fontsize=10,
             color='#4A5A6A', transform=ax2.transAxes, va='top')

    # Live metrics
    metrics = [
        ('Episode reward', f'{total_r:.2f}', '#185FA5'),
        ('Zones covered',  f'{cov}/3',
         '#0F6E56' if cov==3 else '#993C1D' if cov==0 else '#185FA5'),
        ('Redundancies',   str(redundancies),
         '#993C1D' if redundancies>0 else '#0F6E56'),
        ('Step',           f'{step_n}/50', '#4A5A6A'),
    ]
    for j, (label, val, col) in enumerate(metrics):
        y = 0.80 - j*0.13
        ax2.add_patch(mpatches.FancyBboxPatch(
            (0.04, y-0.04), 0.92, 0.10,
            boxstyle='round,pad=0.01',
            facecolor='#FFFFFF', edgecolor='#DDEEFF',
            linewidth=0.8, transform=ax2.transAxes, zorder=1
        ))
        ax2.text(0.08, y+0.02, label, fontsize=9,
                 color='#8899AA', transform=ax2.transAxes, va='center')
        ax2.text(0.96, y+0.02, val, fontsize=13, fontweight='bold',
                 color=col, transform=ax2.transAxes,
                 va='center', ha='right')

    # Finding
    ax2.add_patch(mpatches.FancyBboxPatch(
        (0.04, 0.25), 0.92, 0.18,
        boxstyle='round,pad=0.02',
        facecolor='#E6F1FB' if algo=='PPO' else
                  '#E1F5EE' if algo=='MAPPO' else '#EEEDFE',
        edgecolor=accent, linewidth=1,
        transform=ax2.transAxes, zorder=1
    ))
    ax2.text(0.08, 0.41, 'Research finding:',
             fontsize=9, fontweight='bold',
             color=accent, transform=ax2.transAxes, va='top')
    # Word-wrap
    words = info['finding'].split()
    lines, cur = [], ''
    for w in words:
        if len(cur)+len(w)+1 > 52: lines.append(cur); cur=w
        else: cur=(cur+' '+w).strip()
    if cur: lines.append(cur)
    for k, line in enumerate(lines[:3]):
        ax2.text(0.08, 0.35-k*0.055, line, fontsize=9,
                 color='#4A5A6A', transform=ax2.transAxes, va='top')

    # Phase 1 result
    ax2.add_patch(mpatches.FancyBboxPatch(
        (0.04, 0.06), 0.92, 0.16,
        boxstyle='round,pad=0.02',
        facecolor='#C8F3E0', edgecolor='#0F6E56',
        linewidth=1, transform=ax2.transAxes, zorder=1
    ))
    ax2.text(0.08, 0.19, 'Phase 1 result:',
             fontsize=9, fontweight='bold',
             color='#0F6E56', transform=ax2.transAxes, va='top')
    ax2.text(0.08, 0.13, info['result'],
             fontsize=9, color='#085041',
             transform=ax2.transAxes, va='top')

    # Mode badge
    mode_col = '#534AB7' if state['use_real'] else '#BA7517'
    mode_txt = '🔬 Real model' if state['use_real'] else '⚡ Simulation'
    ax2.text(0.96, 0.97, mode_txt, fontsize=9, fontweight='bold',
             color=mode_col, transform=ax2.transAxes,
             va='top', ha='right')

    plt.tight_layout(pad=1.2)
    with out_canvas:
        clear_output(wait=True)
        plt.show()
    plt.close(fig)

# ── Refresh display ───────────────────────────────────────────
def refresh():
    r = state['runner']
    if r is None or not r.history: return
    snap = r.history[-1]
    step_slider.max   = r.step_n
    step_slider.value = r.step_n
    draw_frame(snap, state['algo'],
               r.step_n, r.total_reward, r.redundancies)

# ── Button callbacks ──────────────────────────────────────────
def on_new_episode(b):
    state['runner'] = EpisodeRunner(
        state['algo'], use_real=state['use_real'])
    refresh()

def on_step(b):
    if state['runner'] is None: on_new_episode(None)
    if not state['runner'].done:
        state['runner'].step()
        refresh()

def on_run_full(b):
    if state['runner'] is None: on_new_episode(None)
    r = state['runner']
    delay = speed_slider.value
    while not r.done:
        r.step()
        refresh()
        time.sleep(delay)

def on_eval(b):
    with out_eval:
        clear_output(wait=True)
        print(f'Running 50 evaluation episodes for {state["algo"]}...')
        rewards = []
        for ep in range(50):
            runner = EpisodeRunner(
                state['algo'], use_real=state['use_real'])
            runner.run_full()
            rewards.append(runner.total_reward)
            if (ep+1)%10==0:
                print(f'  {ep+1}/50 | Mean so far: {np.mean(rewards):.2f}')

        state['episode_rewards'] = rewards
        print(f'\n{"-"*48}')
        print(f'  {state["algo"]} — 50 Episode Evaluation')
        print(f'{"-"*48}')
        print(f'  Mean   : {np.mean(rewards):.2f}')
        print(f'  Std    : {np.std(rewards):.2f}')
        print(f'  Max    : {np.max(rewards):.2f}')
        print(f'  Min    : {np.min(rewards):.2f}')
        print(f'{"-"*48}')

        # Plot
        fig, axes = plt.subplots(1,2,figsize=(11,3.5))
        accent = ALGO_ACCENT[state['algo']]
        axes[0].plot(rewards, alpha=0.4, color=accent, lw=0.8)
        w=10
        if len(rewards)>=w:
            ma=np.convolve(rewards,np.ones(w)/w,mode='valid')
            axes[0].plot(range(w-1,len(rewards)),ma,color=accent,lw=2,
                         label=f'MA{w}: {ma[-1]:.1f}')
        axes[0].axhline(np.mean(rewards),color='#0F6E56',ls='--',
                        label=f'Mean: {np.mean(rewards):.2f}')
        axes[0].set_title(f'{state["algo"]} — Episode rewards',
                          color=accent, fontweight='bold')
        axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

        axes[1].hist(rewards,bins=15,color=accent,edgecolor='white',alpha=0.8)
        axes[1].axvline(np.mean(rewards),color='#0F6E56',ls='--',lw=2,
                        label=f'Mean: {np.mean(rewards):.2f}')
        axes[1].set_title('Reward distribution', fontweight='bold')
        axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)
        plt.tight_layout()
        plt.show()
        plt.close(fig)

def on_algo_change(change):
    state['algo'] = change['new']
    on_new_episode(None)

def on_model_change(change):
    use_real = (change['new'] == 'Real model')
    if use_real and state['algo'] not in REAL_POLICIES:
        with out_info:
            clear_output(wait=True)
            print(f'⚠ Real model not loaded for {state["algo"]}.')
            print('  Set USE_REAL_MODELS=True in Cell 5 and run it first.')
        model_toggle.value = 'Simulation'
        return
    state['use_real'] = use_real
    on_new_episode(None)

# ── Wire up ───────────────────────────────────────────────────
btn_new.on_click(on_new_episode)
btn_step.on_click(on_step)
btn_run.on_click(on_run_full)
btn_eval.on_click(on_eval)
algo_toggle.observe(on_algo_change, names='value')
model_toggle.observe(on_model_change, names='value')

# ── Layout ────────────────────────────────────────────────────
title_html = widgets.HTML(
    '<h2 style="font-family:sans-serif;font-size:18px;font-weight:500;'
    'color:#1E2761;margin:0 0 4px">MARL Coordination Demo</h2>'
    '<p style="font-family:sans-serif;font-size:12px;color:#4A5A6A;margin:0">'
    'simple_spread_v3 · N=3 agents · 3 landmarks · cooperative</p>'
)

controls = widgets.HBox([
    btn_new, btn_step, btn_run,
    widgets.VBox([speed_slider]),
    btn_eval
], layout=widgets.Layout(gap='6px', align_items='center'))

ui = widgets.VBox([
    title_html,
    widgets.HBox([widgets.Label('Algorithm:'), algo_toggle]),
    widgets.HBox([widgets.Label('Mode:'), model_toggle]),
    controls,
    out_canvas,
    out_info,
    out_eval,
], layout=widgets.Layout(gap='6px', width='100%'))

display(ui)

# Start immediately
on_new_episode(None)
print('\n✅ UI launched! Use the buttons above to interact.')
print('   → Algorithm: switch between PPO / MAPPO / MADDPG')
print('   → Step: move one step at a time')
print('   → Run full: play entire episode at chosen speed')
print('   → Eval 50 eps: run 50 episodes and plot results')


✅ UI launched! Use the buttons above to interact.
   → Algorithm: switch between PPO / MAPPO / MADDPG
   → Step: move one step at a time
   → Run full: play entire episode at chosen speed
   → Eval 50 eps: run 50 episodes and plot results
